# MAE3405 Assignment 1 - 2026

Dr Daniel Duke

Department of Mechanical & Aerospace Engineering

Monash University

## Assignment brief

Size a naturally-aspirated engine to meet a specific aircraft's design requirement.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib notebook


## Step 1. Calculate engine Power and BSFC

Assuming an Air-Standard cycle

In [55]:
def standardAtmosphereDensity(z_km):
    r = np.roots((6.10767, -2.20467e1, 1.79402e1-z_km))
    r = r[(r>0)&(r<1.3)]
    return np.min(r)

In [86]:
def boundEfficiency(x): # for scalars
    if x<0: return 0
    elif x>1: return 1
    else: return x  

def fuelAirOttoCycle(T1,rho1,RPM,bore=0.1,stroke=0.1,ncyl=4,compressionRatio=8,\
                     LHV=43.7e3,AFstoich=14.7,equivalenceRatio=1.,combustionEfficiency=1.,\
                    mechanicalEfficiency=.95,fourStroke=True,volumetricEfficiency=1.):
    
    # Calculate the performance of a Fuel-air Otto Cycle.
    # Assumed fuel and air already mixed at intake (non DI)
    # No chemistry calcualtion - provide LHV, AFs & combustion efficiency as inputs.
    # No super/turbocharging calculation - provide T1 and p1 as inputs.
    
    # Temps in Kelvin.
    # Bore, stroke in m
    # LHV in kJ/kg_f

    # Apply some sanity checks to inputs
    mechanicalEfficiency = boundEfficiency(mechanicalEfficiency)
    volumetricEfficiency = boundEfficiency(volumetricEfficiency)
    combustionEfficiency = boundEfficiency(combustionEfficiency)
    assert all((T1>0, rho1>0, bore>0, stroke>0, ncyl>0, compressionRatio>0))
    assert all((LHV>0, AFstoich>0, equivalenceRatio>0))
    
    
    # Thermo constants for Fuel-Air Cycle from notes
    gamma_r = 1.327
    gamma_p = 1.259
    cv_r = 0.8392 #kJ/kg.K
    cv_p = 1.143 #kJ/kg.K

    # Fuel-air cycle
    AF = AFstoich/equivalenceRatio
    Qh = combustionEfficiency * LHV / (1+AF)

    T2 = T1*(compressionRatio)**(gamma_r-1) # isentropic compression
    T3 = (cv_r*(T2-298) + Qh)/cv_p + 298 # isochoric heating
    T4 = T3*(1/compressionRatio)**(gamma_p-1) # isentropic expansion

    eta_th = (cv_p*(T3-T4) - cv_r*(T2-T1))/(cv_p*(T3-298)-cv_r*(T2-298))*combustionEfficiency
    eta_bth = mechanicalEfficiency * eta_th

    # Engine volume and power
    sweptVol = ncyl*stroke*(bore**2)*np.pi*0.25 # m^3
    sweptRate = sweptVol * (RPM/60.) # m^3/s
    if fourStroke: sweptRate /= 2.
    inducedAirRate = rho1*sweptRate*volumetricEfficiency # kg/s
    brakePower = (LHV*1e3/AF)*inducedAirRate*eta_bth # Watts

    # Fuel consumption
    fuelRate = inducedAirRate/AF
    
    # Performance
    BMEP = brakePower/sweptRate/1e6 # MPa
    BSFC = fuelRate / brakePower * 1e3 * 3600 #kg/kW.hr

    return eta_bth, brakePower, BMEP, BSFC
    

In [87]:
def fuelAirDieselCycle(T1,rho1,RPM,bore=0.1,stroke=0.1,ncyl=4,compressionRatio=15,\
                     LHV=42.5e3,AFstoich=14.5,equivalenceRatio=1.,combustionEfficiency=1.,\
                    mechanicalEfficiency=.95,fourStroke=True,volumetricEfficiency=1.,\
                      cutoffRatio=2.6):
    
    # Calculate the performance of a Fuel-air Diesel Cycle.
    # Assumed fuel and air already mixed at intake (non DI)
    # No chemistry calcualtion - provide LHV, AFs & combustion efficiency as inputs.
    # No super/turbocharging calculation - provide T1 and p1 as inputs.
    
    # Temps in Kelvin.
    # Bore, stroke in m
    # LHV in kJ/kg_f

    # Apply some sanity checks to inputs
    mechanicalEfficiency = boundEfficiency(mechanicalEfficiency)
    volumetricEfficiency = boundEfficiency(volumetricEfficiency)
    combustionEfficiency = boundEfficiency(combustionEfficiency)
    assert all((T1>0, rho1>0, bore>0, stroke>0, ncyl>0, compressionRatio>0))
    assert all((LHV>0, AFstoich>0, equivalenceRatio>0))
    
    
    # Thermo constants for Fuel-Air Cycle from notes
    gamma_r = 1.38
    gamma_p = 1.25
    R_r = 0.266 #kJ/kg.K
    R_p = 0.275 #kJ/kg.K
    cp_r = (gamma_r*R_r)/(gamma_r-1) #kJ/kg.K
    cp_p = (gamma_p*R_p)/(gamma_p-1) #kJ/kg.K
    cv_r = (R_r)/(gamma_r-1) #kJ/kg.K
    cv_p = (R_p)/(gamma_p-1) #kJ/kg.K
    
    # Fuel-air cycle
    AF = AFstoich/equivalenceRatio
    Qh = combustionEfficiency * LHV / (1+AF)

    p1 = rho1*R_r*T1*1e3 # Pa
    v1 = 1/rho1 # m3/kg
    
    p2 = p1*(compressionRatio)**(gamma_r)   # isentropic compression
    T2 = T1*(compressionRatio)**(gamma_r-1) # isentropic compression
    v2 = v1/compressionRatio                # isentropic compression
    
    v3 = v2*cutoffRatio                  # isobaric expansion
    w23 = p2*(v3-v2)*1e-3  #kJ/kg        # isobaric expansion
    T3 = (cp_r*(T2-298) + Qh - w23)/cp_p + 298 # isobaric heating
    p3 = R_p*T3/v3

    v4 = v1 # mass and volume back to same as state 1 if non-DI
    p4 = p3*(v3/v4)**gamma_p # note that expansion ratio ≠ compression ratio in Diesel
    T4 = T3*(v3/v4)**(gamma_p-1) # isentropic expansion
    
    Ql = cv_p*(T4-298)-cv_r*(T1-298)
    eta_th = (Qh-Ql)/Qh
    eta_bth = mechanicalEfficiency * eta_th

    # Engine volume and power
    sweptVol = ncyl*stroke*(bore**2)*np.pi*0.25 # m^3
    sweptRate = sweptVol * (RPM/60.) # m^3/s
    if fourStroke: sweptRate /= 2.
    inducedAirRate = rho1*sweptRate*volumetricEfficiency # kg/s
    brakePower = (LHV*1e3/AF)*inducedAirRate*eta_bth # Watts

    # Fuel consumption
    fuelRate = inducedAirRate/AF
    
    # Performance
    BMEP = brakePower/sweptRate/1e6 # MPa
    BSFC = fuelRate / brakePower * 1e3 * 3600 #kg/kW.hr

    return eta_bth, brakePower, BMEP, BSFC

## Step 2. Calculate aircraft performance in steady level flight

Given a particular weight, airspeed, propeller and aerodynamic properties - determine the thrust and power required to fly

In [61]:
from scipy.optimize import root

def propulsiveEfficiency_Froude(airspeed=100., propDiameter=2.0, shaftPower=100e3, rho=1.2):
    # Solve the Froude Theory equations for efficiency and thrust given power input
    
    # Return the efficiency and thrust produced by an ideal propeller
    A = 0.25*np.pi*propDiameter**2
    u = airspeed
    
    def fFroude(vars):
        w, thrust = vars # w is induced velocity m/s
        return [ thrust*(u+w) - shaftPower ,\
                 -2*w - u + np.sqrt(u**2 + 2*thrust/(rho*A)) ]

    # Use static thrust for initial guess
    guess_thrust = np.cbrt( shaftPower**2 * (2*rho*A) )
    guess_w = np.sqrt(guess_thrust/(2*rho*A))
    
    solveFroude = root(fFroude, [guess_w,guess_thrust])
    w, thrust = solveFroude.x
    propulsiveEfficiency=1/(1+w/u)

    return propulsiveEfficiency, thrust

Estimate the range and endurance of a propeller-driven aircraft

In [101]:
def minDragAirspeed(mass, rho, wingArea, dragPolar=(0.02,0.05)): 
    cD0, K = dragPolar # K = 1/(pi*AR*e)
    ustar = np.sqrt(2*mass*9.81/(rho*wingArea)) * (K/cD0)**0.25
    return ustar

def breguetProp(airspeed=50, mass_full=1e3, mass_empty=0.75e3, rho=1.2, dragPolar=(0.02,0.05),\
               propulsiveEfficiency=0.85, BSFC=0.2):
    # Solve range and endurance using the Breguet range equations for a power-producing engine

    # Require lift to balance weight L=mg
    c_L = 0.5*rho*(airspeed**2)/(mass_full*9.81)
    c_D = dragPolar[0] + dragPolar[1]*c_L**2
    glideSlope = c_L/c_D
    
    # Check if c_L in valid range
    infiniteWingAoA = (180/np.pi)*c_L/np.pi # degrees
    if infiniteWingAoA > 12:
        raise RuntimError("stall")

    massRatio = mass_full/mass_empty
    bsfc_si = BSFC * 1e-3 / 3600 # convert kg/kW.hr to kg/Ws

    breguet = propulsiveEfficiency*np.log(massRatio)/(bsfc_si*9.81)
    Eprop = (glideSlope/airspeed)*breguet/3600 # hours
    
    L_DV_best_ratio = np.sqrt(3/4)*(1./3.)**-0.25
    c_L_minDrag = np.sqrt(dragPolar[0]/dragPolar[1])
    c_D_minDrag = 2*dragPolar[0]
    bestGlideSlope = (c_L_minDrag/c_D_minDrag)
    Eprop_max = L_DV_best_ratio*bestGlideSlope*breguet/3600 # hours


    Rprop = glideSlope*breguet/1000 # km
    Rprop_max = bestGlideSlope*breguet/1000 # km
    
    return Eprop, Eprop_max, Rprop, Rprop_max


Determine the absolute ceiling of a naturally-apsirated air-breathing aircraft whose power depends on altitude acccording to

$ W_b = a W_{b,0} + b $

where `altitudePerformance = (a,b)`

In [ ]:
def absCeiling(altitudePerformance=(1.132,0.132),propulsiveEfficiency=0.85,):
    

## Step 3. Optimization

Determine the best values of variables to achieve the target airspeed, ceiling, range and endurance.